# Differentiable Abstraction Search

**Problem**: SoRL's abstraction search (Jacobi recursion) is non-differentiable — we sample discrete tokens, 
so gradients can't flow back through the search process to improve abstraction *selection*.

**Idea (STE-based)**: During the last iteration of Jacobi decoding, replace the hard `argmax`/`multinomial` 
with a Straight-Through Estimator (STE):

```
# Forward: use discrete token index for embedding lookup
# Backward: gradient flows through the soft logits
embedding = embed(sampled_index) + logit - stop_gradient(logit)
```

This means the forward pass still uses discrete tokens (so the model sees normal inputs),
but the backward pass has gradients w.r.t. the logits that produced those tokens.

**This notebook validates the core mechanism on a toy model** before integrating into SoRL:
1. Build a tiny MLP that predicts logits over an abstract vocab
2. Show that STE sampling is differentiable (gradients flow)
3. Show that Gumbel-Softmax is an alternative
4. Compare: hard sampling (no grad) vs STE vs Gumbel-Softmax
5. Train the toy model to optimize a downstream loss *through* the discrete sampling

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Part 1: STE Primitives

Three ways to sample discrete tokens while keeping gradients:

1. **Hard sampling (baseline)** — `multinomial` → no gradient  
2. **STE** — forward uses discrete index, backward uses soft logits: `embed(idx) + logit - sg(logit)`  
3. **Gumbel-Softmax** — differentiable relaxation of categorical sampling

In [ ]:
# ============================================================
# Core primitives: 3 ways to sample discrete tokens
# ============================================================

def sample_hard(logits, temperature=1.0):
    """Standard sampling — NO gradient through the sample."""
    probs = F.softmax(logits / temperature, dim=-1)
    idx = torch.multinomial(probs, num_samples=1).squeeze(-1)  # (B,)
    return idx  # detached from computation graph


def sample_ste(logits, temperature=1.0):
    """
    STE sampling: forward = discrete index, backward = soft logits.
    
    The trick: we want embed(idx) to carry gradients w.r.t. logits.
    
    Step 1: Compute soft probs (differentiable)
    Step 2: Sample hard index (non-differentiable)  
    Step 3: Create one-hot from hard index
    Step 4: STE: one_hot + soft_probs - stop_gradient(soft_probs)
            Forward: one_hot (hard), Backward: soft_probs (differentiable)
    Step 5: Use this "soft one-hot" to do a differentiable embedding lookup
            via matmul with embedding weight matrix
    """
    soft_probs = F.softmax(logits / temperature, dim=-1)       # (B, V) differentiable
    idx = torch.multinomial(soft_probs, num_samples=1).squeeze(-1)  # (B,) hard sample
    one_hot = F.one_hot(idx, num_classes=logits.size(-1)).float()   # (B, V)
    # STE: forward = one_hot, backward = soft_probs
    ste_probs = one_hot + soft_probs - soft_probs.detach()          # (B, V)
    return ste_probs, idx


def sample_gumbel_softmax(logits, temperature=1.0, hard=True):
    """
    Gumbel-Softmax: differentiable relaxation of categorical sampling.
    With hard=True, forward is one-hot but backward is soft.
    """
    soft = F.gumbel_softmax(logits, tau=temperature, hard=hard, dim=-1)  # (B, V)
    idx = soft.argmax(dim=-1)
    return soft, idx


# ---- Verify gradients flow ----
print("=== Gradient flow test ===\n")

V = 8  # small abstract vocab
logits = torch.randn(1, V, requires_grad=True)

# Test 1: Hard sampling — no gradient
idx_hard = sample_hard(logits, temperature=0.5)
# Can't backprop through idx_hard directly, so we use a dummy loss
# that depends on the index — gradient should NOT flow
dummy_embed = nn.Embedding(V, 4)
emb_hard = dummy_embed(idx_hard)
loss_hard = emb_hard.sum()
loss_hard.backward()
print(f"Hard sampling:    logits.grad = {logits.grad}")
# Expected: None (no gradient)

# Test 2: STE — gradient flows
logits2 = torch.randn(1, V, requires_grad=True)
ste_probs, idx_ste = sample_ste(logits2, temperature=0.5)
# Differentiable embedding lookup: matmul with embed weight
emb_ste = ste_probs @ dummy_embed.weight  # (B, 4)
loss_ste = emb_ste.sum()
loss_ste.backward()
print(f"STE sampling:     logits.grad = {logits2.grad is not None} | norm = {logits2.grad.norm().item():.4f}")

# Test 3: Gumbel-Softmax — gradient flows
logits3 = torch.randn(1, V, requires_grad=True)
gumbel_probs, idx_gumbel = sample_gumbel_softmax(logits3, temperature=0.5)
emb_gumbel = gumbel_probs @ dummy_embed.weight
loss_gumbel = emb_gumbel.sum()
loss_gumbel.backward()
print(f"Gumbel-Softmax:   logits.grad = {logits3.grad is not None} | norm = {logits3.grad.norm().item():.4f}")

print("\n✓ STE and Gumbel-Softmax both propagate gradients through discrete sampling.")

## Part 2: Toy Model — Can we optimize *through* discrete token selection?

Setup: A tiny "abstract token predictor" (1-layer MLP) that takes a hidden state and predicts
which abstract token to insert. The downstream loss depends on the *chosen* token's embedding.

**Goal**: Show that STE lets us train the predictor to minimize a downstream loss that only
sees the discrete token — not the soft distribution.

In [ ]:
# ============================================================
# Toy model: MLP predicts abstract token logits from hidden state
# Downstream loss depends on the chosen token's embedding
# ============================================================

class ToyAbstractPredictor(nn.Module):
    """
    Mimics the SoRL abstraction prediction:
    - Takes a hidden state h (like transformer hidden at an abstract position)
    - Predicts logits over abstract vocab
    - Samples a discrete token
    - Looks up that token's embedding
    - Downstream: the embedding is used to predict a target
    """
    def __init__(self, hidden_dim, abs_vocab_size, embed_dim):
        super().__init__()
        self.predictor = nn.Linear(hidden_dim, abs_vocab_size)  # h -> logits
        self.abs_embed = nn.Embedding(abs_vocab_size, embed_dim)  # abstract token embeddings
        self.downstream = nn.Linear(embed_dim, 1)  # downstream task head
        self.abs_vocab_size = abs_vocab_size

    def forward(self, h, temperature=1.0, method="ste"):
        """
        h: (B, hidden_dim)
        Returns: prediction, sampled_idx, logits
        """
        logits = self.predictor(h)  # (B, abs_vocab_size)

        if method == "hard":
            idx = sample_hard(logits, temperature)
            emb = self.abs_embed(idx)  # (B, embed_dim) — NO gradient to logits
        elif method == "ste":
            ste_probs, idx = sample_ste(logits, temperature)
            emb = ste_probs @ self.abs_embed.weight  # (B, embed_dim) — differentiable
        elif method == "gumbel":
            gumbel_probs, idx = sample_gumbel_softmax(logits, temperature)
            emb = gumbel_probs @ self.abs_embed.weight  # (B, embed_dim) — differentiable
        else:
            raise ValueError(f"Unknown method: {method}")

        pred = self.downstream(emb).squeeze(-1)  # (B,)
        return pred, idx, logits


# ---- Toy task: pick the token whose embedding gives output closest to target ----
H = 32       # hidden dim
V_abs = 16   # abstract vocab size
E = 16       # embed dim
B = 64       # batch size

model_toy = ToyAbstractPredictor(H, V_abs, E).to(device)

# Frozen target: for each hidden state h, there's an optimal abstract token
# We'll make the target depend on h so the predictor must learn
target_proj = nn.Linear(H, 1, bias=False).to(device)
for p in target_proj.parameters():
    p.requires_grad = False

print(f"Toy model: hidden={H}, abs_vocab={V_abs}, embed={E}")
print(f"Task: predict target = target_proj(h), choosing abstract tokens as intermediary")
print(f"Methods to compare: hard (no grad), STE, Gumbel-Softmax")

In [ ]:
# ============================================================
# Train toy model with all 3 methods — compare convergence
# ============================================================

def train_toy(method, n_steps=500, lr=1e-3, temperature=0.5):
    """Train the toy abstract predictor with a given sampling method."""
    # Fresh model each time for fair comparison
    mdl = ToyAbstractPredictor(H, V_abs, E).to(device)
    opt = torch.optim.Adam(mdl.parameters(), lr=lr)

    losses = []
    token_entropy = []  # track whether tokens collapse

    for step in range(n_steps):
        h = torch.randn(B, H, device=device)
        target = target_proj(h).squeeze(-1)  # (B,)

        pred, idx, logits = mdl(h, temperature=temperature, method=method)
        loss = F.mse_loss(pred, target.detach())

        opt.zero_grad()
        if method == "hard":
            # REINFORCE-style: can't backprop through hard sampling
            # Use score function estimator as baseline comparison
            with torch.no_grad():
                reward = -loss.item()
            log_probs = F.log_softmax(logits / temperature, dim=-1)
            selected_log_probs = log_probs.gather(1, idx.unsqueeze(1)).squeeze(1)
            reinforce_loss = -(reward * selected_log_probs).mean()
            # Also need downstream loss for the downstream head
            # But downstream head gets no useful gradient through hard sampling
            # So we combine: reinforce for predictor, MSE for downstream
            total_loss = reinforce_loss + loss.detach()  # loss.detach() since hard has no grad path
            total_loss.backward()
        else:
            loss.backward()

        opt.step()
        losses.append(loss.item())

        # Track token diversity
        with torch.no_grad():
            probs = F.softmax(logits / temperature, dim=-1)
            ent = -(probs * (probs + 1e-10).log()).sum(dim=-1).mean().item()
            token_entropy.append(ent)

    return losses, token_entropy


# Run all 3 methods
print("Training with hard sampling (REINFORCE baseline)...")
losses_hard, ent_hard = train_toy("hard", n_steps=500, temperature=0.5)

print("Training with STE...")
losses_ste, ent_ste = train_toy("ste", n_steps=500, temperature=0.5)

print("Training with Gumbel-Softmax...")
losses_gumbel, ent_gumbel = train_toy("gumbel", n_steps=500, temperature=0.5)

print("Done.")

In [ ]:
# ============================================================
# Visualization: Loss convergence + Token diversity
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Loss curves ---
ax = axes[0]
window = 20
def smooth(x, w=window):
    return np.convolve(x, np.ones(w)/w, mode='valid')

ax.plot(smooth(losses_hard), label="Hard (REINFORCE)", alpha=0.8, color="gray")
ax.plot(smooth(losses_ste), label="STE", alpha=0.8, color="tab:blue")
ax.plot(smooth(losses_gumbel), label="Gumbel-Softmax", alpha=0.8, color="tab:orange")
ax.set_xlabel("Step"); ax.set_ylabel("MSE Loss")
ax.set_title("Convergence: Can we optimize through discrete sampling?")
ax.legend(); ax.grid(True, alpha=0.3)

# --- Token entropy (diversity) ---
ax = axes[1]
ax.plot(smooth(ent_hard), label="Hard", alpha=0.8, color="gray")
ax.plot(smooth(ent_ste), label="STE", alpha=0.8, color="tab:blue")
ax.plot(smooth(ent_gumbel), label="Gumbel", alpha=0.8, color="tab:orange")
ax.axhline(y=np.log(V_abs), color='r', linestyle='--', alpha=0.3, label=f'max entropy (uniform)')
ax.set_xlabel("Step"); ax.set_ylabel("Entropy (nats)")
ax.set_title("Token Diversity (higher = more diverse)")
ax.legend(); ax.grid(True, alpha=0.3)

# --- Final loss comparison ---
ax = axes[2]
final_n = 50
methods = ["Hard\n(REINFORCE)", "STE", "Gumbel\nSoftmax"]
final_losses = [np.mean(losses_hard[-final_n:]),
                np.mean(losses_ste[-final_n:]),
                np.mean(losses_gumbel[-final_n:])]
colors = ["gray", "tab:blue", "tab:orange"]
bars = ax.bar(methods, final_losses, color=colors, alpha=0.8)
ax.set_ylabel("Final MSE Loss (last 50 steps)")
ax.set_title("Final Loss Comparison")
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, final_losses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.4f}", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig("./figure/diff_search_toy.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to ./figure/diff_search_toy.png")

## Part 3: STE Applied to Jacobi Recursion (Simulated)

Now simulate the actual SoRL pattern: multiple positions need abstract tokens, 
and the last iteration of Jacobi decoding is made differentiable via STE.

The model does K iterations of refinement. Iterations 1..K-1 are no-grad (normal Jacobi).
**Only the last iteration uses STE** — so the final abstraction choice carries gradients
back through the logits that produced it.

In [ ]:
# ============================================================
# Simulated Jacobi Recursion with STE on last iteration
# ============================================================
# This mimics the SoRL pattern:
#   - A sequence has N_abs abstract positions
#   - Each position gets refined over K iterations (Jacobi)
#   - Iterations 1..K-1: normal hard sampling (no grad)
#   - Iteration K: STE sampling (differentiable)
#   - Downstream: sequence embedding → loss

class JacobiAbstractSearch(nn.Module):
    """
    Simulated Jacobi recursion for abstract token search.
    
    Architecture:
      - context_encoder: encodes surrounding NL tokens → hidden state per abs position
      - abs_predictor: hidden → logits over abstract vocab (refined each iteration)
      - abs_embed: abstract token embeddings
      - downstream: uses full sequence (NL + abstract embeddings) to predict target
    """
    def __init__(self, n_positions, hidden_dim, abs_vocab_size, embed_dim):
        super().__init__()
        self.n_positions = n_positions
        self.abs_vocab_size = abs_vocab_size

        # Per-position context (simulates transformer hidden states at abs positions)
        self.context_encoder = nn.Linear(hidden_dim, hidden_dim)
        # Refiner: takes (context, current_abs_embed) → new logits
        self.refiner = nn.Sequential(
            nn.Linear(hidden_dim + embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, abs_vocab_size),
        )
        self.abs_embed = nn.Embedding(abs_vocab_size, embed_dim)
        # Downstream: aggregate all position embeddings → scalar
        self.downstream = nn.Linear(n_positions * embed_dim, 1)

    def forward(self, contexts, n_iterations=3, temperature=1.0, method="ste"):
        """
        contexts: (B, n_positions, hidden_dim) — context at each abstract position
        Returns: prediction, final_indices, all_logits_history
        """
        B = contexts.shape[0]
        h = self.context_encoder(contexts)  # (B, N, H)

        # Initialize abstract tokens randomly
        current_idx = torch.randint(0, self.abs_vocab_size, (B, self.n_positions), device=contexts.device)
        logits_history = []

        for it in range(n_iterations):
            current_emb = self.abs_embed(current_idx)  # (B, N, E)
            cat = torch.cat([h, current_emb], dim=-1)  # (B, N, H+E)
            logits = self.refiner(cat)                   # (B, N, V_abs)
            logits_history.append(logits.detach())

            is_last = (it == n_iterations - 1)

            if is_last and method == "ste":
                # STE on last iteration — differentiable
                soft_probs = F.softmax(logits / temperature, dim=-1)  # (B, N, V)
                hard_idx = torch.multinomial(
                    soft_probs.view(-1, self.abs_vocab_size), 1
                ).view(B, self.n_positions)  # (B, N)
                one_hot = F.one_hot(hard_idx, self.abs_vocab_size).float()
                ste_probs = one_hot + soft_probs - soft_probs.detach()  # STE
                final_emb = torch.einsum('bnv,ve->bne', ste_probs, self.abs_embed.weight)
                current_idx = hard_idx

            elif is_last and method == "gumbel":
                soft = F.gumbel_softmax(logits, tau=temperature, hard=True, dim=-1)
                final_emb = torch.einsum('bnv,ve->bne', soft, self.abs_embed.weight)
                current_idx = soft.argmax(dim=-1)

            else:
                # Hard sampling (no grad) — all non-last iterations, or method=="hard"
                with torch.no_grad():
                    probs = F.softmax(logits / temperature, dim=-1)
                    current_idx = torch.multinomial(
                        probs.view(-1, self.abs_vocab_size), 1
                    ).view(B, self.n_positions)
                if is_last:
                    final_emb = self.abs_embed(current_idx)  # no gradient to logits

        # Downstream prediction from final abstract embeddings
        flat_emb = final_emb.view(B, -1)  # (B, N*E)
        pred = self.downstream(flat_emb).squeeze(-1)  # (B,)

        return pred, current_idx, logits_history


N_POS = 4   # number of abstract positions
H = 32
V_abs = 16
E = 16

jacobi_model = JacobiAbstractSearch(N_POS, H, V_abs, E).to(device)
print(f"Jacobi model: {N_POS} positions, {V_abs} vocab, {3} iterations")
print(f"Only last iteration is differentiable (STE/Gumbel)")

In [ ]:
# ============================================================
# Train Jacobi model with all 3 methods — compare convergence
# ============================================================

# Target: a fixed random projection from contexts → scalar
jacobi_target = nn.Linear(N_POS * H, 1, bias=False).to(device)
for p in jacobi_target.parameters():
    p.requires_grad = False


def train_jacobi(method, n_steps=800, lr=1e-3, temperature=0.5, n_iters=3):
    mdl = JacobiAbstractSearch(N_POS, H, V_abs, E).to(device)
    opt = torch.optim.Adam(mdl.parameters(), lr=lr)
    losses = []
    token_counts = []  # track unique tokens chosen

    for step in range(n_steps):
        contexts = torch.randn(B, N_POS, H, device=device)
        target = jacobi_target(contexts.view(B, -1)).squeeze(-1)

        pred, idx, _ = mdl(contexts, n_iterations=n_iters,
                           temperature=temperature, method=method)
        loss = F.mse_loss(pred, target.detach())

        opt.zero_grad()
        if method == "hard":
            # REINFORCE for predictor gradients
            with torch.no_grad():
                reward = -loss.item()
            # Re-forward to get logits with grad for REINFORCE
            h = mdl.context_encoder(contexts)
            current_emb = mdl.abs_embed(idx)
            cat = torch.cat([h, current_emb], dim=-1)
            logits = mdl.refiner(cat)
            log_probs = F.log_softmax(logits / temperature, dim=-1)
            selected_lp = log_probs.gather(2, idx.unsqueeze(-1)).squeeze(-1)
            reinforce_loss = -(reward * selected_lp).mean()
            reinforce_loss.backward()
        else:
            loss.backward()

        opt.step()
        losses.append(loss.item())

        # Token diversity
        unique = len(idx.unique())
        token_counts.append(unique)

    return losses, token_counts


print("Training Jacobi model (3 iterations, STE on last)...")
print("  Method: hard (REINFORCE)...")
j_losses_hard, j_tc_hard = train_jacobi("hard", n_steps=800, temperature=0.5)
print("  Method: STE...")
j_losses_ste, j_tc_ste = train_jacobi("ste", n_steps=800, temperature=0.5)
print("  Method: Gumbel-Softmax...")
j_losses_gumbel, j_tc_gumbel = train_jacobi("gumbel", n_steps=800, temperature=0.5)
print("Done.")

In [ ]:
# ============================================================
# Visualization: Jacobi recursion comparison
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
window = 30

ax = axes[0]
ax.plot(smooth(j_losses_hard, window), label="Hard (REINFORCE)", alpha=0.8, color="gray")
ax.plot(smooth(j_losses_ste, window), label="STE (last iter)", alpha=0.8, color="tab:blue")
ax.plot(smooth(j_losses_gumbel, window), label="Gumbel (last iter)", alpha=0.8, color="tab:orange")
ax.set_xlabel("Step"); ax.set_ylabel("MSE Loss")
ax.set_title("Jacobi Recursion: Loss Convergence")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(smooth(j_tc_hard, window), label="Hard", alpha=0.8, color="gray")
ax.plot(smooth(j_tc_ste, window), label="STE", alpha=0.8, color="tab:blue")
ax.plot(smooth(j_tc_gumbel, window), label="Gumbel", alpha=0.8, color="tab:orange")
ax.axhline(y=V_abs, color='r', linestyle='--', alpha=0.3, label=f'max ({V_abs})')
ax.set_xlabel("Step"); ax.set_ylabel("Unique tokens / batch")
ax.set_title("Token Diversity")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.axis('off')
final_n = 50
summary = (
    f"Jacobi Recursion Results (3 iters)\n"
    f"{'='*36}\n\n"
    f"Final MSE (last {final_n} steps):\n"
    f"  Hard (REINFORCE): {np.mean(j_losses_hard[-final_n:]):.4f}\n"
    f"  STE:              {np.mean(j_losses_ste[-final_n:]):.4f}\n"
    f"  Gumbel:           {np.mean(j_losses_gumbel[-final_n:]):.4f}\n\n"
    f"Token diversity (last {final_n} steps):\n"
    f"  Hard:   {np.mean(j_tc_hard[-final_n:]):.1f} / {V_abs}\n"
    f"  STE:    {np.mean(j_tc_ste[-final_n:]):.1f} / {V_abs}\n"
    f"  Gumbel: {np.mean(j_tc_gumbel[-final_n:]):.1f} / {V_abs}\n\n"
    f"If STE/Gumbel converge faster with\n"
    f"similar diversity -> diff search works."
)
ax.text(0.05, 0.5, summary, transform=ax.transAxes, fontsize=11,
        verticalalignment='center', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.tight_layout()
plt.savefig("./figure/diff_search_jacobi.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to ./figure/diff_search_jacobi.png")

## Part 4: SoRL Integration Sketch

How this would plug into the real `SoRLTrainerv3._training_step()`:

```python
# In sorl_wrapper.py — recursion() modified:
def recursion_diff(self, idx, attention_mask, max_iterations=5, temperature=1.0, ...):
    """Jacobi recursion with STE on the last iteration."""
    for it in range(max_iterations):
        outputs = self.model.forward(input_ids=idx, attention_mask=attention_mask, ...)
        logits = outputs.logits
        
        if it < max_iterations - 1:
            # Normal hard sampling (no grad)
            idx = self.extract_and_sample(logits, idx, recursion_mask, temperature)
        else:
            # Last iteration: STE
            # For each abstract position, replace hard index with STE embedding
            abs_logits = logits[:, abs_positions, vocab_size_0:]  # (B, N_abs, V_abs)
            soft_probs = F.softmax(abs_logits / temperature, dim=-1)
            hard_idx = torch.multinomial(soft_probs.view(-1, V_abs), 1).view(B, N_abs)
            one_hot = F.one_hot(hard_idx, V_abs).float()
            ste_probs = one_hot + soft_probs - soft_probs.detach()
            # This ste_probs can be used in the loss computation
            # instead of looking up embeddings by index
    
    return idx, per_token_loss, ste_probs  # ste_probs carries gradient
```

Key insight: only the **last iteration** needs STE. Earlier iterations are exploratory
and can use standard hard sampling. This keeps the cost identical to current Jacobi.

In [ ]:
# ============================================================
# Part 5: Temperature sweep — how does temperature affect STE?
# ============================================================
# In SoRL, temperature controls exploration. For STE, it also
# affects gradient magnitude (low temp → sharper softmax → larger grads).
# Let's see the trade-off.

temps = [0.1, 0.3, 0.5, 1.0, 2.0]
results_by_temp = {}

for temp in temps:
    print(f"  temp={temp}...")
    losses, tc = train_jacobi("ste", n_steps=500, temperature=temp, n_iters=3)
    results_by_temp[temp] = {"losses": losses, "tc": tc}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
for temp in temps:
    ax.plot(smooth(results_by_temp[temp]["losses"], 20),
            label=f"temp={temp}", alpha=0.8)
ax.set_xlabel("Step"); ax.set_ylabel("MSE Loss")
ax.set_title("STE: Temperature vs Convergence")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
for temp in temps:
    ax.plot(smooth(results_by_temp[temp]["tc"], 20),
            label=f"temp={temp}", alpha=0.8)
ax.axhline(y=V_abs, color='r', linestyle='--', alpha=0.3)
ax.set_xlabel("Step"); ax.set_ylabel("Unique tokens / batch")
ax.set_title("STE: Temperature vs Token Diversity")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./figure/diff_search_temp.png", dpi=150, bbox_inches='tight')
plt.show()

# Summary
print("\nFinal loss by temperature:")
for temp in temps:
    fl = np.mean(results_by_temp[temp]["losses"][-50:])
    ft = np.mean(results_by_temp[temp]["tc"][-50:])
    print(f"  temp={temp}: loss={fl:.4f}, diversity={ft:.1f}/{V_abs}")